# Module 3 · Chapter 2 — t · χ² · F 분포: 왜 필요한가

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 세 분포가 등장하는 이유, 정규에서 파생되는 구조
- **이 노트북**: 시뮬레이션으로 각 분포를 생성하고 그 관계를 눈으로 확인

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. t 분포 — 자유도에 따라 꼬리가 얼마나 두꺼워지나

자유도(df)가 커질수록 t 분포는 표준 정규 N(0,1)에 수렴합니다.

In [ ]:
x = np.linspace(-5, 5, 500)
dfs = [1, 3, 10, 30]
colors = ["tomato", "orange", "steelblue", "mediumseagreen"]

fig, ax = plt.subplots(figsize=(9, 4))

for df_val, color in zip(dfs, colors):
    ax.plot(x, stats.t.pdf(x, df=df_val), color=color,
            linewidth=2, label=f"t(df={df_val})")

ax.plot(x, stats.norm.pdf(x), color="black",
        linewidth=2, linestyle="--", label="N(0,1)")

ax.set_title("자유도가 커질수록 t 분포 → 표준 정규")
ax.set_xlabel("x")
ax.set_ylabel("밀도")
ax.set_ylim(0, 0.45)
ax.legend()
plt.tight_layout()
plt.show()

# 임계값 비교
alpha = 0.05
print("=== 95% 신뢰구간 임계값 비교 (양측, α=0.05) ===")
print(f"{'자유도':<10} {'t 임계값':>12} {'차이(vs z)':<12}")
z_crit = stats.norm.ppf(1 - alpha/2)
for df_val in [3, 5, 10, 20, 50, 100]:
    t_crit = stats.t.ppf(1 - alpha/2, df=df_val)
    print(f"{df_val:<10} {t_crit:>12.4f} {t_crit - z_crit:>10.4f}")
print(f"{'z(∞)':<10} {z_crit:>12.4f}")

## 2. t 분포의 탄생을 시뮬레이션으로

정규 모집단에서 표본을 뽑아 T-통계량을 계산하면 실제로 t 분포가 만들어집니다.

In [ ]:
n_sample = 10  # 소표본
true_mu  = 5.0
true_sigma = 1.5
n_reps = 10_000

# T-통계량 계산: T = (x̄ - μ) / (s / √n)
t_stats = []
for _ in range(n_reps):
    sample = rng.normal(true_mu, true_sigma, n_sample)
    t_val  = (sample.mean() - true_mu) / (sample.std(ddof=1) / np.sqrt(n_sample))
    t_stats.append(t_val)

t_stats = np.array(t_stats)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(t_stats, bins=80, density=True, alpha=0.6, color="steelblue",
        label="시뮬레이션 T-통계량")

x_line = np.linspace(-5, 5, 300)
ax.plot(x_line, stats.t.pdf(x_line, df=n_sample-1),
        color="tomato", linewidth=2, label=f"t(df={n_sample-1}) 이론값")
ax.plot(x_line, stats.norm.pdf(x_line),
        color="gray", linewidth=1.5, linestyle="--", label="N(0,1)")

ax.set_title(f"소표본(n={n_sample})에서 T-통계량 분포 (반복 {n_reps:,}회)")
ax.set_xlabel("T")
ax.set_ylabel("밀도")
ax.legend()
ax.set_xlim(-5, 5)
plt.tight_layout()
plt.show()

## 3. χ² 분포 — 표준 정규 변수의 제곱합

Z₁² + Z₂² + ... + Zₖ² ~ χ²(k)

In [ ]:
n_reps = 50_000
dfs_chi = [1, 2, 5, 10]
colors_chi = ["tomato", "orange", "steelblue", "mediumseagreen"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽: 이론 PDF
x_chi = np.linspace(0.01, 20, 400)
for df_val, color in zip(dfs_chi, colors_chi):
    axes[0].plot(x_chi, stats.chi2.pdf(x_chi, df=df_val),
                 color=color, linewidth=2, label=f"χ²(df={df_val})")
axes[0].set_title("χ² 분포 PDF (이론)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("밀도")
axes[0].set_ylim(0, 0.6)
axes[0].legend()

# 오른쪽: 시뮬레이션으로 χ²(5) 재현
df_sim = 5
z_samples = rng.standard_normal((n_reps, df_sim))
chi2_sim  = (z_samples ** 2).sum(axis=1)  # 각 행의 제곱합

axes[1].hist(chi2_sim, bins=80, density=True, alpha=0.6,
             color="steelblue", label=f"시뮬레이션 (df={df_sim})")
axes[1].plot(x_chi, stats.chi2.pdf(x_chi, df=df_sim),
             color="tomato", linewidth=2, label=f"χ²({df_sim}) 이론값")
axes[1].set_title(f"Z²합 시뮬레이션 → χ²(df={df_sim}) 검증")
axes[1].set_xlabel("x")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. F 분포 — 두 χ² 변수의 비

In [ ]:
df1, df2 = 5, 10
n_reps = 50_000

# F = (χ²₁/df1) / (χ²₂/df2)
chi2_1 = rng.chisquare(df=df1, size=n_reps)
chi2_2 = rng.chisquare(df=df2, size=n_reps)
f_sim  = (chi2_1 / df1) / (chi2_2 / df2)

x_f = np.linspace(0.01, 8, 400)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(f_sim, bins=100, density=True, alpha=0.6,
        color="steelblue", label=f"시뮬레이션 (df1={df1}, df2={df2})")
ax.plot(x_f, stats.f.pdf(x_f, dfn=df1, dfd=df2),
        color="tomato", linewidth=2, label=f"F({df1},{df2}) 이론값")
ax.set_title("χ² 비로 만든 F 분포 검증")
ax.set_xlabel("F")
ax.set_ylabel("밀도")
ax.set_xlim(0, 8)
ax.legend()
plt.tight_layout()
plt.show()

print("임상 시나리오에서의 F 해석:")
print("F = (집단 간 분산) / (집단 내 분산)")
print("F가 클수록 집단 평균 차이가 우연 이상임을 시사 → ANOVA (Module 8에서)")

## 5. 임상 시나리오 데이터로 세 통계량 계산

아직 통계적 검정은 하지 않습니다 — '얼마인지'를 보는 것이 이 챕터의 목표입니다.

In [ ]:
# 두 그룹 회복 시간 (일)
group_a = rng.normal(loc=8.0, scale=2.0, size=15)  # 치료군
group_b = rng.normal(loc=10.0, scale=2.5, size=15) # 대조군

# T-통계량 (수식으로 직접)
mean_a, mean_b = group_a.mean(), group_b.mean()
s_a, s_b = group_a.std(ddof=1), group_b.std(ddof=1)
n_a, n_b = len(group_a), len(group_b)
pooled_se = np.sqrt(s_a**2/n_a + s_b**2/n_b)
t_stat = (mean_a - mean_b) / pooled_se

print("=== T-통계량 (평균 비교) ===")
print(f"치료군 평균: {mean_a:.2f}일,  대조군 평균: {mean_b:.2f}일")
print(f"T = {t_stat:.4f}")

# χ²-통계량 (관측 빈도 vs 기대 빈도)
# 부작용 종류: 없음/경증/중증
observed = np.array([8, 5, 2])   # 치료군
expected = np.array([10, 3, 2])  # 사전 예상 비율 기반
chi2_stat = ((observed - expected)**2 / expected).sum()
print("\n=== χ²-통계량 (범주형 적합도) ===")
print(f"관측: {observed},  기대: {expected}")
print(f"χ² = {chi2_stat:.4f}")

# F-통계량 (분산 비교)
f_stat = (s_a**2) / (s_b**2)  # 두 그룹 분산 비
print("\n=== F-통계량 (분산 비교) ===")
print(f"치료군 분산: {s_a**2:.4f},  대조군 분산: {s_b**2:.4f}")
print(f"F = {f_stat:.4f}")
print("\n※ 이 통계량들의 p-value 해석은 Module 6(가설검정)에서 다룹니다.")

## 6. 직접 답해 보기

1. 자유도가 10인 t-분포와 표준 정규 N(0,1)의 95% 임계값 차이는 얼마인가? 왜 차이가 생기는가?
2. χ² 분포가 왜 항상 0 이상의 값만 가지는지 수식으로 설명하면?
3. F 분포의 자유도가 두 개(df1, df2)인 이유는 무엇인가?